
# ЛР №4. Аналитическая витрина Seoul Bike

**Вопрос:** Как меняется связь температуры со спросом на велопрокат в зависимости от времени суток?

**Гранулярность:** одна строка - один час конкретной даты.

**Ключ:** сочетание `event_date` и `hour_of_day`.

Для анализа используются температура, количество аренд и час суток. Нерабочие часы сохраняются с флагом качества, но в дальнейшем могут исключаться из расчёта показателей.

Используется одна широкая таблица без дополнительных справочников.


In [1]:
import os
from pathlib import Path

import boto3
import duckdb
from dotenv import load_dotenv

root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file()),
    None
)

load_dotenv(root / ".env", override=True)

staging_path = "s3://staging/seoul-bike-sharing-demand/source_ingested_on=2026-09-30/staging.parquet"
mart_key = "seoul-bike-sharing-demand/source_ingested_on=2026-09-30/hourly_rentals.parquet"
mart_path = f"s3://mart/{mart_key}"

con = duckdb.connect()
con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")

endpoint = os.environ["S3_ENDPOINT"].removeprefix("http://").removeprefix("https://")

con.execute(f"SET s3_endpoint = '{endpoint}'")
con.execute(f"SET s3_access_key_id = '{os.environ['S3_ACCESS_KEY']}'")
con.execute(f"SET s3_secret_access_key = '{os.environ['S3_SECRET_KEY']}'")
con.execute("SET s3_url_style = 'path'")
con.execute("SET s3_use_ssl = false")


In [2]:
con.execute(f"""
    CREATE OR REPLACE TEMP VIEW staging AS
    SELECT *
    FROM read_parquet('{staging_path}', hive_partitioning = false)
""")

fields = [
    "event_date",
    "hour_of_day",
    "rented_bike_count",
    "temperature_c",
    "functioning_day",
    "quality_non_functioning_hour_violated"
]


schema = con.execute("DESCRIBE staging").fetchall()

available_fields = {row[0] for row in schema}

assert set(fields).issubset(available_fields)

print("Поля staging:")
for row in schema:
    print(row[0], row[1])

result = con.execute("""
    SELECT
        count(*) AS rows_count,
        count(*) FILTER (
            WHERE quality_non_functioning_hour_violated
        ) AS flagged_rows
    FROM staging
""").fetchone()

print("Всего строк:", result[0])
print("Строк с флагом:", result[1])


Поля staging:
event_date DATE
hour_of_day INTEGER
rented_bike_count BIGINT
temperature_c DOUBLE
season VARCHAR
functioning_day VARCHAR
quality_non_functioning_hour_violated BOOLEAN
Всего строк: 8760
Строк с флагом: 295


## 2. Сборка витрины

Витрина `hourly_rentals` содержит почасовые наблюдения спроса на велопрокат и температуры воздуха.

Все строки из staging сохраняются без фильтрации. Нерабочие часы остаются в таблице вместе с флагами качества.


In [3]:

con.execute("""
    CREATE OR REPLACE TEMP TABLE hourly_rentals AS
    SELECT
        event_date,
        hour_of_day,
        rented_bike_count,
        temperature_c,
        functioning_day,
        quality_non_functioning_hour_violated
    FROM staging
""")

rows = con.execute("""
    SELECT count(*)
    FROM hourly_rentals
""").fetchone()[0]

assert rows == 8760

print("Строк в витрине:", rows)

con.execute("""
    SELECT *
    FROM hourly_rentals
    ORDER BY event_date, hour_of_day
    LIMIT 5
""").fetchall()


Строк в витрине: 8760


[(datetime.date(2017, 12, 1), 0, 254, -5.2, 'Yes', False),
 (datetime.date(2017, 12, 1), 1, 204, -5.5, 'Yes', False),
 (datetime.date(2017, 12, 1), 2, 173, -6.0, 'Yes', False),
 (datetime.date(2017, 12, 1), 3, 107, -6.2, 'Yes', False),
 (datetime.date(2017, 12, 1), 4, 78, -6.0, 'Yes', False)]

## 3. Сохранение витрины

Витрина сохраняется в формате Parquet в бакет mart. При повторном выполнении предыдущий файл заменяется.


In [4]:
s3_endpoint = os.environ["S3_ENDPOINT"]

s3 = boto3.client(
    "s3",
    endpoint_url=s3_endpoint if s3_endpoint.startswith("http") else f"http://{s3_endpoint}",
    aws_access_key_id=os.environ["S3_ACCESS_KEY"],
    aws_secret_access_key=os.environ["S3_SECRET_KEY"],
    region_name="us-east-1"
)

s3.delete_object(Bucket="mart", Key=mart_key)

con.execute(f"""
    COPY hourly_rentals TO '{mart_path}'
    (FORMAT PARQUET)
""")

file_info = s3.head_object(Bucket="mart", Key=mart_key)

print("Файл сохранён:", mart_path)
print("Размер файла:", file_info["ContentLength"], "байт")


Файл сохранён: s3://mart/seoul-bike-sharing-demand/source_ingested_on=2026-09-30/hourly_rentals.parquet
Размер файла: 81608 байт


## 4. Проверка сохранённой витрины

Читаем Parquet из бакета mart. Проверяем количество строк, схему, составной ключ, значения и флаги качества.


In [5]:
con.execute(f"""
    CREATE OR REPLACE TEMP VIEW saved_mart AS
    SELECT *
    FROM read_parquet('{mart_path}', hive_partitioning = false)
""")

schema = con.execute("DESCRIBE saved_mart").fetchall()

print("Схема витрины:")
for row in schema:
    print(row[0], row[1])

result = con.execute("""
    SELECT count(*)
    FROM saved_mart
""").fetchone()[0]

print("Количество строк:", result)


Схема витрины:
event_date DATE
hour_of_day INTEGER
rented_bike_count BIGINT
temperature_c DOUBLE
functioning_day VARCHAR
quality_non_functioning_hour_violated BOOLEAN
Количество строк: 8760


In [6]:
result = con.execute("""
    SELECT
        (SELECT count(*) FROM staging) AS staging_rows,
        (SELECT count(*) FROM saved_mart) AS mart_rows,
        (SELECT count(*)
         FROM saved_mart
         WHERE event_date IS NULL
            OR hour_of_day IS NULL) AS null_keys,
        (SELECT count(*)
         FROM (
             SELECT event_date, hour_of_day
             FROM saved_mart
             GROUP BY event_date, hour_of_day
             HAVING count(*) > 1
         )) AS duplicate_keys
""").fetchone()

print("Строк в staging:", result[0])
print("Строк в mart:", result[1])
print("Пропусков в ключе:", result[2])
print("Повторяющихся ключей:", result[3])


Строк в staging: 8760
Строк в mart: 8760
Пропусков в ключе: 0
Повторяющихся ключей: 0


In [7]:
check = con.execute("""
    SELECT
        count(*) FILTER (WHERE rented_bike_count IS NULL) AS null_rentals,
        count(*) FILTER (WHERE temperature_c IS NULL) AS null_temperature,
        count(*) FILTER (WHERE functioning_day IS NULL) AS null_functioning_day,
        count(*) FILTER (WHERE quality_non_functioning_hour_violated IS NULL) AS null_flags,
        count(*) FILTER (WHERE hour_of_day NOT BETWEEN 0 AND 23) AS invalid_hours,
        count(*) FILTER (WHERE rented_bike_count < 0) AS negative_rentals,
        min(rented_bike_count) AS min_rentals,
        max(rented_bike_count) AS max_rentals,
        min(temperature_c) AS min_temperature,
        max(temperature_c) AS max_temperature
    FROM saved_mart
""")

values = check.fetchone()

for field, value in zip([col[0] for col in check.description], values):
    print(field, value)


null_rentals 0
null_temperature 0
null_functioning_day 0
null_flags 0
invalid_hours 0
negative_rentals 0
min_rentals 0
max_rentals 3556
min_temperature -17.8
max_temperature 39.4


In [8]:
result = con.execute("""
    SELECT
        (SELECT count(*) FROM staging
         WHERE quality_non_functioning_hour_violated) AS staging_flags,

        (SELECT count(*) FROM saved_mart
         WHERE quality_non_functioning_hour_violated) AS mart_flags,

        (SELECT count(*) FROM saved_mart
         WHERE functioning_day = 'No') AS non_functioning,

        (SELECT count(*) FROM saved_mart
         WHERE quality_non_functioning_hour_violated
         IS DISTINCT FROM (functioning_day = 'No')) AS flag_errors
""").fetchone()

print("Флагов в staging:", result[0])
print("Флагов в mart:", result[1])
print("Нерабочих часов:", result[2])
print("Ошибок флагов:", result[3])


Флагов в staging: 295
Флагов в mart: 295
Нерабочих часов: 295
Ошибок флагов: 0


In [9]:
columns = ", ".join(fields)

missing = con.execute(f"""
    SELECT count(*)
    FROM (
        SELECT {columns} FROM staging
        EXCEPT ALL
        SELECT {columns} FROM saved_mart
    )
""").fetchone()[0]

extra = con.execute(f"""
    SELECT count(*)
    FROM (
        SELECT {columns} FROM saved_mart
        EXCEPT ALL
        SELECT {columns} FROM staging
    )
""").fetchone()[0]

print("Отсутствующих строк:", missing)
print("Лишних или изменённых строк:", extra)


Отсутствующих строк: 0
Лишних или изменённых строк: 0


## 5. Проверка аналитического среза

Проверяем распределение работающих часов проката по времени суток.

Нерабочие часы исключаются только из контрольного запроса. В сохранённой витрине остаются все исходные наблюдения.


In [10]:
result = con.execute("""
    SELECT
        hour_of_day,
        count(*) AS observations
    FROM saved_mart
    WHERE functioning_day = 'Yes'
    GROUP BY hour_of_day
    ORDER BY hour_of_day
""").fetchall()

for hour, count in result:
    print(hour, count)

print("Количество групп:", len(result))
print("Всего наблюдений:", sum(row[1] for row in result))


0 352
1 352
2 352
3 352
4 352
5 352
6 352
7 353
8 353
9 353
10 353
11 353
12 353
13 353
14 353
15 353
16 353
17 353
18 353
19 353
20 353
21 353
22 353
23 353
Количество групп: 24
Всего наблюдений: 8465
